# Búsqueda A* — 8-Puzzle

Lee `estadoinicial.txt` y `estadofinal.txt`. El vacío se representa con **0**.

## Heurística inventada: *Eco decimal de columnas*

Cada columna del tablero se lee de arriba hacia abajo como un número de tres dígitos.  
Por ejemplo, la columna izquierda de:

```
3 2 6
5 0 4
1 8 7
```

vale **351**.

La heurística compara esas tres cifras con las del estado meta:

$$
h = \frac{|C_0 - C_0^*| + |C_1 - C_1^*| + |C_2 - C_2^*|}{100}
$$

Si las columnas se parecen a las del objetivo, $h$ baja.  
A* usa $f = g + h$, donde $g$ es la cantidad de movimientos hechos.

In [ ]:
# Cola de prioridad (mínimo primero). En A* la usamos para abrir
# siempre el nodo con menor f = g + h.
try:
    import heapq
except ImportError:
    %pip install heapq
    import heapq


## Lectura y utilidades

In [ ]:
def leer_estado(ruta):
    """Lee un archivo .txt y lo convierte en un tablero 3x3.

    Ejemplo de archivo:
        326
        504
        187
    Devuelve una tupla de tuplas, p.ej. ((3,2,6), (5,0,4), (1,8,7)).
    Usamos tuplas porque son inmutables y sirven como clave en diccionarios.
    """
    tablero = []
    with open(ruta) as f:
        for linea in f:
            linea = linea.strip()          # quita espacios y saltos de línea
            if linea:                     # ignora líneas vacías
                # "326" -> (3, 2, 6)
                tablero.append(tuple(int(c) for c in linea))
    return tuple(tablero)                 # ((3,2,6), (5,0,4), (1,8,7))


def mostrar(estado):
    """Imprime el tablero en pantalla, una fila por línea."""
    for fila in estado:
        print(*fila)   # el * desempaca: (3,2,6) se imprime como 3 2 6
    print()


def buscar_cero(estado):
    """Devuelve (fila, columna) donde está el espacio vacío (0)."""
    for i in range(3):
        for j in range(3):
            if estado[i][j] == 0:
                return i, j

## Heurística

In [ ]:
def heuristica(estado, meta):
    """Estima qué tan lejos está 'estado' de 'meta' (sin buscar el camino).

    Eco decimal de columnas:
      - Lee cada columna de arriba a abajo como un número de 3 dígitos.
        Ej: columna [3,5,1] -> 3*100 + 5*10 + 1 = 351
      - Suma |número_actual - número_meta| de las 3 columnas.
      - Divide entre 100 para que h no sea un número enorme.

    Mientras más parecidas sean las columnas al objetivo, más bajo es h.
    """
    h = 0
    for j in range(3):  # j = 0,1,2  (columna izquierda, centro, derecha)
        # convierte la columna j del estado actual en un número de 3 dígitos
        col_act = estado[0][j] * 100 + estado[1][j] * 10 + estado[2][j]
        # lo mismo para la meta
        col_meta = meta[0][j] * 100 + meta[1][j] * 10 + meta[2][j]
        h += abs(col_act - col_meta)   # diferencia absoluta entre ambas
    return h / 100

## Vecinos y A*

In [ ]:
# Posibles movimientos del hueco: (cambio_fila, cambio_col, nombre)
# fila -1 = subir una fila, col +1 = moverse a la derecha, etc.
DIRS = [
    (-1, 0, "Arriba"),
    (1, 0, "Abajo"),
    (0, -1, "Izquierda"),
    (0, 1, "Derecha"),
]


def vecinos(estado):
    """Genera todos los tableros a los que se puede llegar en 1 movimiento.

    Mueve el hueco (0) en las 4 direcciones, si no se sale del tablero.
    Devuelve una lista de pares: (nuevo_tablero, nombre_del_movimiento).
    """
    i, j = buscar_cero(estado)   # posición actual del hueco
    resultado = []
    for di, dj, nombre in DIRS:
        ni, nj = i + di, j + dj          # nueva posición del hueco
        if 0 <= ni < 3 and 0 <= nj < 3:  # ¿sigue dentro del tablero 3x3?
            # copia el tablero a listas (las tuplas no se pueden modificar)
            nuevo = [list(fila) for fila in estado]
            # intercambia el hueco con la ficha vecina
            nuevo[i][j], nuevo[ni][nj] = nuevo[ni][nj], nuevo[i][j]
            # vuelve a tuplas y guarda el resultado
            resultado.append((tuple(tuple(f) for f in nuevo), nombre))
    return resultado


def a_estrella(inicio, meta):
    """Busca el camino del estado inicial al final con A*.

    Ideas clave:
      g = costo real (cuántos movimientos llevamos)
      h = estimación (heurística) de lo que falta
      f = g + h  → cuanto más bajo, más prometedor es el nodo

    Retorna:
      (lista_de_movimientos, iteraciones) si hay solución
      (None, iteraciones) si se agota la búsqueda sin encontrar meta
    """
    # Cola de prioridad: siempre se saca el nodo con menor f
    # Cada elemento es (f, g, estado)
    cola = [(heuristica(inicio, meta), 0, inicio)]

    # vino_de[estado] = (estado_padre, movimiento_que_llegó_aquí)
    # Sirve para reconstruir el camino al final. El inicio no tiene padre.
    vino_de = {inicio: None}

    # costo_g[estado] = mejor g conocido para llegar a ese estado
    costo_g = {inicio: 0}

    iteraciones = 0  # cuántas veces sacamos un nodo de la cola

    while cola:
        # Saca el nodo más prometedor (menor f)
        f, g, actual = heapq.heappop(cola)
        iteraciones += 1

        # ¿Llegamos a la meta?
        if actual == meta:
            # Reconstruye el camino andando hacia atrás con vino_de
            movs = []
            e = actual
            while vino_de[e] is not None:
                padre, mov = vino_de[e]
                movs.append(mov)
                e = padre
            movs.reverse()  # quedó al revés (meta → inicio), lo volteamos
            return movs, iteraciones

        # Si ya conocíamos un camino más barato a 'actual', ignoramos este
        if g > costo_g[actual]:
            continue

        # Expande: prueba cada movimiento posible desde 'actual'
        for nxt, mov in vecinos(actual):
            nuevo_g = g + 1  # un movimiento más
            # Si nunca visitamos nxt, o llegamos con menor costo, lo abrimos
            if nxt not in costo_g or nuevo_g < costo_g[nxt]:
                costo_g[nxt] = nuevo_g
                vino_de[nxt] = (actual, mov)  # recordamos de dónde viene
                # f = g + h
                heapq.heappush(cola, (nuevo_g + heuristica(nxt, meta), nuevo_g, nxt))

    # La cola se vació sin alcanzar la meta
    return None, iteraciones

## Ejecución

In [ ]:
# Carga los dos tableros desde los archivos de texto
inicio = leer_estado("estadoinicial.txt")
meta = leer_estado("estadofinal.txt")

print("Estado inicial:")
mostrar(inicio)
print("Estado final:")
mostrar(meta)
# Valor de la heurística en el punto de partida (qué tan "lejos" se ve)
print("h(inicio) =", heuristica(inicio, meta))

In [ ]:
# Ejecuta A* y obtiene la lista de movimientos + cuántas iteraciones usó
movimientos, iteraciones = a_estrella(inicio, meta)

if movimientos is None:
    print("No se encontró solución.")
    print("Iteraciones:", iteraciones)
else:
    print("Iteraciones:", iteraciones)                    # nodos sacados de la cola
    print("Cantidad de movimientos:", len(movimientos))  # largo del camino solución
    print()
    print("Movimientos del hueco:")
    for i, m in enumerate(movimientos, 1):
        print(f"  {i}. {m}")

    # Replay: aplica cada movimiento sobre el tablero para ver la trayectoria
    print()
    print("Trayectoria:")
    estado = inicio
    print("Paso 0")
    mostrar(estado)
    for i, m in enumerate(movimientos, 1):
        # busca entre los vecinos el que corresponde al movimiento m
        for nxt, nombre in vecinos(estado):
            if nombre == m:
                estado = nxt
                break
        print(f"Paso {i}: {m}")
        mostrar(estado)